In [ ]:
import pandas as pd

df = pd.read_csv("../data/loans_sample.csv.gz", low_memory=False)
df.shape
df["loan_status"].value_counts()

bad_statuses = ["Charged Off", "Default"]
good_statuses = ["Fully Paid"]

loan_status
Fully Paid                                             32316
Current                                                26287
Charged Off                                             8036
Late (31-120 days)                                       671
In Grace Period                                          293
Late (16-30 days)                                        131
Does not meet the credit policy. Status:Fully Paid        65
Does not meet the credit policy. Status:Charged Off       19
Default                                                    3
Name: count, dtype: int64

In [ ]:
# Columns that were only known AFTER the loan was issued
LEAKAGE_COLS = [
    # Payments made after the loan started
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    # Credit info pulled after the loan started
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    # Flags that only exist once a borrower is struggling
    'pymnt_plan', 'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
    'debt_settlement_flag', 'debt_settlement_flag_date', 'settlement_status',
    'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term',
]

# Junk you can't catch with rules: IDs, links, free text, duplicates
JUNK_COLS = ['id', 'member_id', 'url', 'desc', 'title', 'emp_title', 'policy_code']

FAIRNESS_COLS = ['zip_code']

def find_junk(df, max_missing=0.9, protect=()):
    junk = [c for c in JUNK_COLS if c in df.columns]
    for col in df.columns:
        if col in junk or col in protect:
            continue
        if df[col].nunique(dropna=True) <= 1:     # same value in every row, e.g. policy_code
            junk.append(col)
        elif df[col].isna().mean() > max_missing: # over 90% empty
            junk.append(col)
    return junk

buckets['leakage'] = [c for c in LEAKAGE_COLS if c in df.columns]
buckets['junk'] = [c for c in find_junk(df, protect=['loan_status'])
                   if c not in buckets['leakage']]
buckets['fairness'] = [c for c in FAIRNESS_COLS if c in df.columns]

df_clean = df.drop(columns=buckets['leakage'] + buckets['junk'] + buckets['fairness'])
## print(len(buckets['leakage']), len(buckets['junk']), len(buckets['fairness']), df_clean.shape)
df_clean.shape


(67821, 89)

In [14]:
import pandas as pd

# File read
df = pd.read_csv("../data/loans_sample.csv.gz", low_memory=False)
##print("Loaded:", df.shape)

# Keeping Individual applications only
df = df[df["application_type"] == "Individual"].copy()
##print("Individual only:", df.shape)

# Leakage: only known AFTER the loan was issued
LEAKAGE_COLS = [
    # Payments made after the loan started
    "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    # Credit info pulled after the loan started
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low",
    # Flags that only exist once a borrower is struggling
    "pymnt_plan", "hardship_flag", "hardship_type", "hardship_reason", "hardship_status",
    "deferral_term", "hardship_amount", "hardship_start_date", "hardship_end_date",
    "payment_plan_start_date", "hardship_length", "hardship_dpd", "hardship_loan_status",
    "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
    "debt_settlement_flag", "debt_settlement_flag_date", "settlement_status",
    "settlement_date", "settlement_amount", "settlement_percentage", "settlement_term",
]

# Junk: IDs, links, free text, duplicates, fairness risk
JUNK_COLS = ["id", "member_id", "url", "desc", "title", "emp_title", "zip_code"]

def find_junk(df, max_missing=0.9, protect=()):
    junk = [c for c in JUNK_COLS if c in df.columns]
    for col in df.columns:
        if col in junk or col in protect:
            continue
        if df[col].nunique(dropna=True) <= 1:       
            junk.append(col)
        elif df[col].isna().mean() > max_missing:   
            junk.append(col)
    return junk

leakage = [c for c in LEAKAGE_COLS if c in df.columns]
junk = [c for c in find_junk(df, protect=["loan_status"]) if c not in leakage]

##print(f"\nLeakage dropped ({len(leakage)})")
##print(f"Junk dropped ({len(junk)}):", junk)

df_clean = df.drop(columns=leakage + junk)
##print("\nAfter dropping:", df_clean.shape)

# Fixing str cols
DATE_COLS = ["issue_d", "earliest_cr_line"]
for col in DATE_COLS:
    df_clean[col] = pd.to_datetime(df_clean[col], format="%b-%Y", errors="coerce")

# " 60 months" -> 60
df_clean["term"] = df_clean["term"].str.extract(r"(\d+)")[0].astype(float)

# "< 1 year" -> 0, "10+ years" -> 10, "3 years" -> 3
df_clean["emp_length"] = (
    df_clean["emp_length"]
    .replace({"< 1 year": "0", "10+ years": "10"})
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

CATEGORY_COLS = df_clean.select_dtypes(include=["object", "string"]).columns
for col in CATEGORY_COLS:
    df_clean[col] = df_clean[col].astype("category")

##print("Final:", df_clean.shape)
print("Category columns:", list(CATEGORY_COLS))

bad_statuses = ["Charged Off", "Default"]
good_statuses = ["Fully Paid"]

df_model = df_clean[df_clean["loan_status"].isin(bad_statuses + good_statuses)].copy()
df_model["target"] = df_model["loan_status"].isin(bad_statuses).astype(int)

df_model["loan_status"] = df_model["loan_status"].cat.remove_unused_categories()


df_model = df_model[
    (df_model["issue_d"].dt.year <= 2015) & (df_model["term"] == 36)
].copy()

print(df_model.shape)
print(df_model["target"].value_counts())
print(df_model["target"].mean())



Category columns: ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'initial_list_status', 'disbursement_method']
(18557, 89)
target
0    15971
1     2586
Name: count, dtype: int64
0.13935442151209787
